# ショット雑音（Shot Noise）トイモデル

ショット雑音は「電荷が離散的に到着する」ことに由来する揺らぎです。
到着イベントをポアソン過程とみなすと、各サンプル区間に入るイベント数がランダムになり、電流（または電荷流束）が揺らぎます。

## トイモデル
- 到着イベント: ポアソン過程（到着率 `λ` [1/s]）
- 1イベントあたりの電荷: `q` [C]
- 離散化: サンプル区間 `dt` ごとに到着数を数えて電流に換算（`i_raw = q*k/dt`）

このノートでは、電流の“生”の系列 `i_raw(t)` と、その PSD を可視化します。
（実際の観測では帯域制限で PSD が `|H(f)|^2` で整形されますが、ここではフィルタ処理は入れません。）

## 見るポイント
- `λ` を上げるほどイベントが密になり、相対的な揺らぎが小さく見える
- `q` を変えると振幅（分散）がスケールする
- PSD は（低周波側の有限長・平均との差し引き等を除けば）概ねフラットに見える（白色に近い）

（※定量式の厳密な一致より、見た目の理解を優先した簡易モデルです。）


In [2]:
import numpy as np
import matplotlib.pyplot as plt

# ===== パラメータ =====
fs = 50_000.0       # サンプリング周波数 [Hz]
T = 0.5             # 観測時間 [s]
N = int(fs * T)
dt = 1.0 / fs

lam = 2_000.0       # 到着率 λ [1/s]
q = 1.0             # 1イベントあたりの電荷（任意単位でも可）

rng = np.random.default_rng(0)

# ===== イベント生成（各サンプル内の到着数 ~ Poisson(lam*dt)） =====
k = rng.poisson(lam * dt, size=N)

i_raw = q * k / dt  # 各ビンに入った電荷を電流に換算

t = dt * np.arange(N)

# ===== プロット（時間波形） =====
plt.figure(figsize=(10, 4))
m = int(0.02 * N)  # 最初の 2% だけ拡大表示
plt.plot(t[:m] * 1e3, i_raw[:m], lw=1.0, label="raw (binned Poisson)")
plt.xlabel("t [ms]")
plt.ylabel("current [arb.]")
plt.title("Shot noise toy model: Poisson arrivals → current")
plt.grid(True)
plt.legend()
plt.show()

# ===== PSD（簡易 Welch） =====
def welch_psd(x, fs, nperseg=4096, noverlap=2048):
    x = np.asarray(x)
    step = nperseg - noverlap
    if step <= 0:
        raise ValueError("noverlap must be < nperseg")
    w = np.hanning(nperseg)
    scale = fs * (w**2).sum()
    psd = None
    count = 0
    for start in range(0, len(x) - nperseg + 1, step):
        seg = x[start : start + nperseg]
        seg = seg - seg.mean()
        X = np.fft.rfft(seg * w)
        P = (np.abs(X) ** 2) / scale
        psd = P if psd is None else (psd + P)
        count += 1
    psd = psd / max(count, 1)
    f = np.fft.rfftfreq(nperseg, d=1.0 / fs)
    return f, psd

f, P_raw = welch_psd(i_raw, fs)

plt.figure(figsize=(7, 5))
plt.loglog(f[1:], P_raw[1:], label="raw")
plt.xlabel("f [Hz]")
plt.ylabel("PSD [arb.^2/Hz]")
plt.title("PSD (Welch, qualitative)")
plt.grid(True, which="both")
plt.legend()
plt.show()


ModuleNotFoundError: No module named 'matplotlib'